This notebook demonstrates how to run a Fisher forecast on difffky. 

We will compute gradients of the likelihood with respect to *bounded* parameters and we will not evaluate any target data: we assume the predicetd data exactly matches the observed data. With these assumptions, the computation of the Fisher matrix simplifies to:

$$ F = \frac{1}{\lambda} \frac{\partial \lambda}{\partial \theta_i}\frac{\partial \lambda}{\partial \theta_j} ~,$$

where $\lambda$ is the predicted data, i.e. the sum of the number of galaxies per bin in the differentiable histogram. This expression is obtained assuming the Poisson likelihood.

In [ ]:
%cd /home/nvilla/diffstuff/diffsky

# Set `dir_out`

In [ ]:
dir_out = '/home/nvilla/diffstuff/experiments/hmc_dev/scripts_hmc/output'

# Get LC data

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from dsps.cosmology import flat_wcdm
from dsps.data_loaders import load_ssp_templates, load_transmission_curve
from diffsky.experimental.lc_generators.lc_data_phot import weighted_lc_data_phot
from diffsky.param_utils import diffsky_param_wrapper_merging as dpwm

from diffsky.experimental.inference import utils, likelihood, dictionaries

ran_key = jax.random.key(42)

In [ ]:
# -- Settings --

num_halos = 500
z_min = 0.1
z_max = 0.2
lgmp_min = 10.5
lgmp_max = 15.0
sky_area_degsq = 1

ran_key, lc_data_key = jax.random.split(ran_key, 2)

# Cosmology
cosmo_params = flat_wcdm.PLANCK15
fb = 0.156

n_z_phot_table = 15

# Transmission curves data
filter_names = ["u", "g", "r", "i", "z"]
tcurves_args = {
    "fn": None,
    "bn_pat": "sdss_{}_transmission.h5",
    "drn": "/home/nvilla/diffstuff/data/filters",
}

# SSP data arguments
ssp_data_args = {
    "fn": None,
    "drn": "/home/nvilla/diffstuff/data/dsps_drn",
    "bn": "fsps_v0.4.7_mist_c3k_a_kroupa_wNE_logGasU-2.0_logGasZ0.0.h5",
}



# -- Get lc_data_phot --

# Load SSP data
ssp_data = load_ssp_templates(**ssp_data_args)

# Load transmission curve data (wavelength array and transmission)
bn_list = [tcurves_args["bn_pat"].format(x) for x in filter_names]
tcurves = [
    load_transmission_curve(
        fn=tcurves_args["fn"], bn_pat=bn, drn=tcurves_args["drn"]
    )
    for bn in bn_list
]

# Define a redshift table used for photometry interpolation
z_phot_table = jnp.linspace(z_min, z_max, n_z_phot_table)

# Get weighted LC data
lc_data_phot = weighted_lc_data_phot(
    lc_data_key,
    num_halos,
    z_min,
    z_max,
    lgmp_min,
    lgmp_max,
    sky_area_degsq=sky_area_degsq,
    ssp_data=ssp_data,
    tcurves=tcurves,
    z_phot_table=z_phot_table,
    cosmo_params=cosmo_params,
    logmp_cutoff=11.0,
)

gal_weight = lc_data_phot.halo_weight

# Load diffsky parameters

In [ ]:
# Get a list with names of unbounded parameters
var_uparams_list = dictionaries.PARAM_SUBSETS['diffstarpop_means']

# Convert the names into bounded parameters names (just removing u_)
var_params_list = [utils.bounded_name(v) for v in var_uparams_list]

In [ ]:
# Evaluation state: ParamCollection
param_coll = dpwm.DEFAULT_PARAM_COLLECTION
# Evaluation state: DiffskyParamsFlat
param_flat = dpwm.unroll_param_collection_into_flat_array(*param_coll)

#var_params_list = param_flat._fields[16:88]
#var_params_list = ['mean_ulgm_mseq_ytp', 'mean_ulgy_mseq_ytp', 'mean_ulgm_qseq_ytp', 'mean_ulgy_qseq_ytp', 'std_ulgm_mseq_int', 'std_ulgy_mseq_int', 'std_ulgm_qseq_int', 'std_ulgy_qseq_int']

# Evaluation state: varied, flat, bounded
if var_params_list is None:
    var_param_flat = param_flat
else:
    var_param_flat = utils.get_var_param_flat_from_param_flat(param_flat, var_params_list)

# Define data vector function

This function computes only the predicted diff. histogram. It does not compare it with the observed diff. hist.

In [ ]:
from diffsky.experimental.lc_generators.lc_phot import mc_lc_phot

def target_space_fn(mags):
    return mags[:, 2]


def diff_hist_from_param_coll(param_coll, loss_data):
    
    lc_data, XBINS, loss_key = loss_data
    loss_key, phot_key = jax.random.split(loss_key, 2)

    # Get phot. lightcone
    phot_kern_results, phot_randoms, merging_randoms = mc_lc_phot(
        phot_key,
        lc_data,
        mc_merge=0,
        param_collection=param_coll,
    )
    pred_mags = phot_kern_results.obs_mags_weighted

    # Compute pred. diff. hist
    pred_data = target_space_fn(pred_mags)
    # gal_weight = lc_data.halo_weight
    # gal_weight_masked = get_masked_gal_weight(pred_mags, gal_weight)
    XHIST_PRED = likelihood.soft_xhist(pred_data, XBINS)

    return XHIST_PRED

In [ ]:
NBINS = 30
XBOUNDS = (10.0, 40.0)
XBINS = np.linspace(*XBOUNDS, NBINS)[:-1]

In [ ]:
ran_key, loss_key = jax.random.split(ran_key, 2)
loss_data = lc_data_phot, XBINS, loss_key

## Flat version

In [ ]:
def flat_diff_hist_fn(var_param_flat, diffsky_params, loss_data):
    
    param_coll = utils.get_param_coll_from_var_param_flat(
        var_param_flat, diffsky_params
    )
    
    return diff_hist_from_param_coll(param_coll, loss_data)

# Compute gradients and Fisher matrix

In [ ]:
import time
from jax.flatten_util import ravel_pytree

def _jvp_jacobian(eval_point, diff_hist, verbose=True, **kwargs):
    num_var_params = len(eval_point)
    eval_point_flat, unflatten = ravel_pytree(eval_point)

    @jax.jit
    def jac_column(p, e_i, kw):
        def diff_hist_flat(p_flat):
            return diff_hist(unflatten(p_flat), **kw)
        
        # JAX computes the derivative of diff_hist_flat strictly w.r.t. p
        _, jcol = jax.jvp(diff_hist_flat, (p,), (e_i,))
        return jcol

    eye = jnp.eye(num_var_params)

    start = time.time()
    cols = []
    for i in range(num_var_params):
        col_i = jnp.asarray(jac_column(eval_point_flat, eye[i], kwargs))
        jax.block_until_ready(col_i)
        cols.append(col_i)
        if verbose:
            print(
                f"  jvp column {i + 1}/{num_var_params} done ({time.time() - start:.1f}s)"
            )
    return jnp.stack(cols, axis=1)  # (n_bins, n_params)

In [ ]:
# compute data vector
pred_diff_hist = jnp.asarray(flat_diff_hist_fn(var_param_flat, param_flat, loss_data))
inv_pred = 1.0 / pred_diff_hist

# compute jacobian
jac = _jvp_jacobian(eval_point=var_param_flat, diff_hist=flat_diff_hist_fn, diffsky_params=param_flat, loss_data=loss_data)
fim = jac.T @ (jac * inv_pred[:, None])

print("inverting fisher matrix -> covariance matrix")
covariance_matrix = jnp.linalg.inv(fim)

# Correlation matrix

In [ ]:
std_devs = np.sqrt(np.diag(covariance_matrix))
corr_matrix = covariance_matrix / np.outer(std_devs, std_devs)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(corr_matrix, vmin=-1.0, vmax=1.0)
plt.colorbar()
plt.show()